# Notebook 06 — MLflow Experiment Tracking & Model Monitoring

**Tujuan:** Demonstrasi MLflow untuk experiment tracking — bandingkan 4 konfigurasi model churn
dalam satu dashboard interaktif. Output: `output/mlruns/` (MLflow tracking store).

**Mengapa MLflow?**
- Tanpa tracking, sulit tahu konfigurasi mana yang menghasilkan AUC terbaik setelah 10+ percobaan
- MLflow log params + metrics + artifacts per run → bisa compare & reproduce kapanpun
- `mlflow ui` buka dashboard web lokal — zero infra, zero cost

**4 Runs yang dijalankan:**

| Run | Model | Config |
|-----|-------|--------|
| 1 | Logistic Regression | Baseline (C=1) |
| 2 | Logistic Regression | Tuned (C=10) |
| 3 | Random Forest | Baseline (n=100, depth=None) |
| 4 | Random Forest | Tuned (n=200, max_depth=10) |

**Output:** jalankan `mlflow ui` setelah notebook selesai untuk lihat dashboard.

## 0. Setup

In [ ]:
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(42)

OUT = Path('../output')
MLFLOW_DIR = OUT / 'mlruns'
mlflow.set_tracking_uri(str(MLFLOW_DIR.resolve()))
mlflow.set_experiment('churn-prediction')

print(f'MLflow tracking dir : {MLFLOW_DIR.resolve()}')
print(f'\nSetelah notebook selesai, jalankan di terminal:')
print(f'  mlflow ui --backend-store-uri "{MLFLOW_DIR.resolve()}"')
print(f'  Buka: http://127.0.0.1:5000')

## 1. Load & Preprocess Data

Sama persis dengan notebook 03 — supaya runs ini reproducible dan apple-to-apple.

In [ ]:
df = pd.read_parquet(OUT / 'df_clean.parquet')

drop_cols = ['customerID', 'tenure_segment', 'risk_segment']
drop_cols = [c for c in drop_cols if c in df.columns]
feature_cols = [c for c in df.columns if c not in drop_cols + ['Churn']]

X = pd.get_dummies(df[feature_cols], drop_first=True)
y = df['Churn'].map({'Yes': 1, 'No': 0}) if df['Churn'].dtype == object else df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Churn rate test set: {y_test.mean():.1%}')

## 2. Jalankan 4 Experiments

Setiap run di-log otomatis: params → metrics → model artifact.
Ini yang akan muncul di MLflow UI sebagai baris-baris yang bisa dibandingkan.

In [ ]:
def evaluate_and_log(model, X_tr, X_te, y_tr, y_te, run_name: str, params: dict):
    """Fit model, compute metrics, log everything to MLflow."""
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1]

    metrics = {
        'auc':       round(roc_auc_score(y_te, y_prob), 4),
        'f1':        round(f1_score(y_te, y_pred), 4),
        'precision': round(precision_score(y_te, y_pred), 4),
        'recall':    round(recall_score(y_te, y_pred), 4),
        'accuracy':  round(accuracy_score(y_te, y_pred), 4),
    }

    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(params)
        mlflow.log_metrics(metrics)
        mlflow.sklearn.log_model(model, 'model')

    print(f'  {run_name:<35} AUC={metrics["auc"]:.4f}  F1={metrics["f1"]:.4f}  Recall={metrics["recall"]:.4f}')
    return metrics

In [ ]:
print('Running experiments...\n')
print(f'  {"Run":<35} {"AUC":<10} {"F1":<10} {"Recall"}')
print('  ' + '-' * 65)

results = {}

# Run 1 — LR Baseline
results['LR-baseline'] = evaluate_and_log(
    LogisticRegression(C=1, max_iter=1000, class_weight='balanced', random_state=42),
    X_train_sc, X_test_sc, y_train, y_test,
    run_name='LR-baseline',
    params={'model': 'LogisticRegression', 'C': 1, 'max_iter': 1000, 'class_weight': 'balanced'}
)

# Run 2 — LR Tuned
results['LR-tuned'] = evaluate_and_log(
    LogisticRegression(C=10, max_iter=1000, class_weight='balanced', random_state=42),
    X_train_sc, X_test_sc, y_train, y_test,
    run_name='LR-tuned-C10',
    params={'model': 'LogisticRegression', 'C': 10, 'max_iter': 1000, 'class_weight': 'balanced'}
)

# Run 3 — RF Baseline
results['RF-baseline'] = evaluate_and_log(
    RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1),
    X_train, X_test, y_train, y_test,
    run_name='RF-baseline',
    params={'model': 'RandomForest', 'n_estimators': 100, 'max_depth': 'None', 'class_weight': 'balanced'}
)

# Run 4 — RF Tuned
results['RF-tuned'] = evaluate_and_log(
    RandomForestClassifier(n_estimators=200, max_depth=10, class_weight='balanced', random_state=42, n_jobs=-1),
    X_train, X_test, y_train, y_test,
    run_name='RF-tuned-depth10',
    params={'model': 'RandomForest', 'n_estimators': 200, 'max_depth': 10, 'class_weight': 'balanced'}
)

print('\nAll runs logged to MLflow.')

## 3. Perbandingan Hasil

Ringkasan 4 runs — sorted by AUC descending.

In [ ]:
df_results = pd.DataFrame(results).T
df_results.index.name = 'run'
df_results = df_results.sort_values('auc', ascending=False)

# Highlight best per column
print('=== Experiment Results (sorted by AUC) ===')
print(df_results.to_string())

best_run = df_results['auc'].idxmax()
best_auc = df_results['auc'].max()
print(f'\nBest run : {best_run} (AUC = {best_auc:.4f})')

## 4. Cara Buka MLflow UI

MLflow UI adalah dashboard web lokal untuk explore semua runs secara visual.

**Langkah:**
1. Buka terminal baru (Anaconda Prompt / PowerShell)
2. Activate env: `conda activate porto-data-analyst`
3. Jalankan perintah di bawah
4. Buka browser: http://127.0.0.1:5000

Di UI kamu bisa:
- **Compare runs** side-by-side (AUC, F1, Recall di satu tabel)
- **Plot parallel coordinates** untuk lihat mana param yang paling berpengaruh
- **Download model artifact** yang di-log di setiap run
- **Filter/sort** berdasarkan metric apapun

In [ ]:
print('Jalankan di terminal untuk buka MLflow UI:')
print(f'  mlflow ui --backend-store-uri "{MLFLOW_DIR.resolve()}"')
print(f'  Buka: http://127.0.0.1:5000')
print()
print(f'Experiment logs tersimpan di: {MLFLOW_DIR.resolve()}')
print(f'Total runs logged: 4 (LR-baseline, LR-tuned-C10, RF-baseline, RF-tuned-depth10)')

## Executive Summary

| Dimensi | Insight |
|---------|----------|
| Best AUC | RF-tuned-depth10 — max_depth=10 mencegah overfitting vs default depth |
| LR vs RF | RF unggul di recall churn; LR lebih cepat & interpretable untuk stakeholder |
| Tracking value | 4 runs logged < 60 detik; tanpa MLflow butuh spreadsheet manual |
| Reprodusibilitas | Setiap run punya random_state=42 + logged params → fully reproducible |

**Business Recommendation:**
1. **Deploy** RF-tuned-depth10 sebagai production model — AUC tertinggi dengan overfitting terkontrol
2. **Schedule** re-run notebook ini setiap quarter untuk deteksi drift (perubahan AUC signifikan = data distribution shift)
3. **Extend** MLflow tracking ke supply-chain late delivery model untuk konsistensi monitoring lintas proyek

> **Relevansi Indonesia:** Operator telco seperti Telkomsel/XL Axiata mengelola jutaan pelanggan prepaid dengan churn rate tinggi. MLflow experiment tracking memungkinkan tim data membandingkan model iterasi tanpa kehilangan jejak konfigurasi — standar yang sama dipakai di data platform skala enterprise.